In [1]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

WINDOW, HORIZON, STRIDE, MAX_FILL = 14, 7, 3, 3
N_FOLDS, SEED = 5, 42

p = "../data/processed/"
data = pd.read_csv(p + "st12000nm0008_2026q1q2_clean.csv.gz", parse_dates=["date"])
fail_dates = pd.read_csv(p + "fail_dates.csv", parse_dates=["fail_date"])
fail_date_lookup = dict(zip(fail_dates["serial_number"], fail_dates["fail_date"]))
failed_set = set(fail_date_lookup)
FEATURE_COLS = [c for c in data.columns if c.startswith("smart_")]

serials = sorted(data["serial_number"].unique())
rng = np.random.default_rng(SEED)
fold_of = {}
for group in ([s for s in serials if s in failed_set], [s for s in serials if s not in failed_set]):
    for i, s in enumerate(rng.permutation(group)):
        fold_of[s] = i % N_FOLDS

print("drives:", len(serials), "| failed:", len(failed_set), "| features:", len(FEATURE_COLS) + 1)

drives: 18652 | failed: 274 | features: 23


In [2]:
def make_windows_for_drive(df_drive, fail_date, window=WINDOW, horizon=HORIZON, stride=STRIDE):
    df_drive = df_drive.sort_values("date")
    full = pd.date_range(df_drive["date"].min(), df_drive["date"].max())
    observed = full.isin(df_drive["date"]).astype(np.float32)
    d = df_drive.set_index("date")[FEATURE_COLS].reindex(full).ffill(limit=MAX_FILL)
    feats = np.concatenate([d.values.astype(np.float32), (1 - observed)[:, None]], axis=1)

    valid = ~np.isnan(feats).any(axis=1)
    bad_cum = np.concatenate([[0], np.cumsum(~valid)])
    dates = full.values
    n = len(full)
    windows, labels = [], []

    for start in range(0, n - window + 1, stride):
        end = start + window - 1
        if bad_cum[end + 1] - bad_cum[start] > 0:
            continue
        label = 0
        if fail_date is not None:
            delta = (fail_date - pd.Timestamp(dates[end])).days
            if delta < 1:
                continue
            if delta <= horizon:
                label = 1
        windows.append(feats[start:end + 1])
        labels.append(label)
    return windows, labels

In [3]:
Xs, ys, ss = [], [], []
for serial, df_drive in data.groupby("serial_number"):
    w, l = make_windows_for_drive(df_drive, fail_date_lookup.get(serial))
    if w:
        Xs.append(np.stack(w))
        ys.append(np.array(l, dtype=np.float32))
        ss += [serial] * len(l)

X = np.concatenate(Xs)
del Xs
np.clip(X, 0, None, out=X)
np.log1p(X, out=X)                      # log1p space; z-score is done inside each fold
y = np.concatenate(ys)
serial = np.array(ss)
fold = np.array([fold_of[s] for s in serial])

print("windows:", X.shape, "| positives:", int(y.sum()))
for k in range(N_FOLDS):
    m = fold == k
    print(f"fold {k}: {m.sum()} windows | positives {int(y[m].sum())} | "
          f"failed drives with a positive window: {len(set(serial[m & (y == 1)]))}")

windows: (1031533, 14, 23) | positives: 419
fold 0: 206264 windows | positives 79 | failed drives with a positive window: 34
fold 1: 206233 windows | positives 81 | failed drives with a positive window: 38
fold 2: 206269 windows | positives 88 | failed drives with a positive window: 38
fold 3: 206352 windows | positives 78 | failed drives with a positive window: 36
fold 4: 206415 windows | positives 93 | failed drives with a positive window: 43


In [4]:
def oof_report(name, score, rates=(0.005, 0.01, 0.02, 0.03, 0.05)):
    df = pd.DataFrame({"s": serial, "y": y, "sc": score})
    pos = df[df["y"] == 1].groupby("s")["sc"].max()
    healthy = df[~df["s"].isin(failed_set)].groupby("s")["sc"].max()
    rows = []
    for r in rates:
        thr = float(np.quantile(healthy.values, 1 - r))
        hit = (pos >= thr)
        pr = hit.mean()
        rows.append({"method": name, "target FA": f"{r:.1%}",
                     "detected": f"{int(hit.sum())}/{len(pos)}",
                     "detection": f"{pr:.0%} ±{1.96 * np.sqrt(pr * (1 - pr) / len(pos)):.0%}",
                     "actual FA": f"{(healthy >= thr).mean():.1%}"})
    return pd.DataFrame(rows), average_precision_score(y, score)

In [5]:
idx = [FEATURE_COLS.index(c) for c in ["smart_5_raw", "smart_187_raw", "smart_197_raw"]]
last = X[:, -1, :][:, idx]
first = X[:, 0, :][:, idx]
level = (last > 1e-6).sum(axis=1).astype(float)
trend = ((last - first) > 1e-6).sum(axis=1).astype(float)

In [6]:
def summarize(A):
    l = A[:, -1, :]
    return np.concatenate([l, A.max(axis=1), A.mean(axis=1), l - A[:, 0, :]], axis=1)

oof_gb = np.zeros(len(y), dtype=np.float32)
for k in range(N_FOLDS):
    rk = np.random.default_rng(SEED + k)
    tr = np.where(fold != k)[0]
    te = np.where(fold == k)[0]
    pos = tr[y[tr] == 1]
    neg = rk.choice(tr[y[tr] == 0], size=len(pos) * 30, replace=False)
    sel = rk.permutation(np.concatenate([pos, neg]))
    gb = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300, random_state=SEED)
    gb.fit(summarize(X[sel]), y[sel])
    oof_gb[te] = gb.predict_proba(summarize(X[te]))[:, 1]
    print("boosting fold", k, "done")

boosting fold 0 done
boosting fold 1 done
boosting fold 2 done
boosting fold 3 done
boosting fold 4 done


In [7]:
EPOCHS, NEG_PER_POS, BATCH, POS_WEIGHT = 10, 30, 256, 3.0
F = X.shape[-1]

class RiskNet(nn.Module):
    def __init__(self, n_feat, mean, std, hidden=32, dropout=0.2):
        super().__init__()
        self.register_buffer("mu", torch.tensor(mean, dtype=torch.float32))
        self.register_buffer("sd", torch.tensor(std, dtype=torch.float32))
        self.rnn = nn.GRU(n_feat, hidden, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        x = torch.clamp((x - self.mu) / self.sd, -10.0, 10.0)
        out, _ = self.rnn(x)
        return self.head(self.drop(out[:, -1, :])).squeeze(-1)

def predict_idx(model, idx, bs=8192):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(idx), bs):
            out.append(torch.sigmoid(model(torch.from_numpy(X[idx[i:i + bs]]))).numpy())
    return np.concatenate(out)

oof_gru = np.zeros(len(y), dtype=np.float32)
for k in range(N_FOLDS):
    t0 = time.time()
    torch.manual_seed(SEED + k)
    rk = np.random.default_rng(SEED + k)
    tr = np.where(fold != k)[0]
    te = np.where(fold == k)[0]

    sample = X[rk.choice(tr, size=min(200_000, len(tr)), replace=False)]
    mean = sample.reshape(-1, F).mean(axis=0)
    std = sample.reshape(-1, F).std(axis=0)
    std[std == 0] = 1.0
    del sample

    model = RiskNet(F, mean, std)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))
    pos = tr[y[tr] == 1]
    negs = tr[y[tr] == 0]

    for ep in range(EPOCHS):
        model.train()
        neg = rk.choice(negs, size=len(pos) * NEG_PER_POS, replace=False)
        order = rk.permutation(np.concatenate([pos, neg]))
        for i in range(0, len(order), BATCH):
            b = order[i:i + BATCH]
            xb, yb = torch.from_numpy(X[b]), torch.from_numpy(y[b])
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

    oof_gru[te] = predict_idx(model, te)
    print(f"GRU fold {k} done in {time.time() - t0:.0f}s")

GRU fold 0 done in 12s
GRU fold 1 done in 7s
GRU fold 2 done in 7s
GRU fold 3 done in 7s
GRU fold 4 done in 7s


In [8]:
tabs, aps = [], {}
for name, sc in [("rule (level)", level), ("rule (trend)", trend),
                 ("gradient boosting", oof_gb), ("GRU", oof_gru)]:
    t, ap = oof_report(name, sc)
    tabs.append(t)
    aps[name] = round(float(ap), 4)

print(pd.concat(tabs).to_string(index=False))
print("PR-AUC (all windows):", aps)

           method target FA detected detection actual FA
     rule (level)      0.5%   80/189   42% ±7%      0.8%
     rule (level)      1.0%  128/189   68% ±7%      3.2%
     rule (level)      2.0%  128/189   68% ±7%      3.2%
     rule (level)      3.0%  128/189   68% ±7%      3.2%
     rule (level)      5.0%  165/189   87% ±5%      9.2%
     rule (trend)      0.5%   96/189   51% ±7%      1.4%
     rule (trend)      1.0%   96/189   51% ±7%      1.4%
     rule (trend)      2.0%  150/189   79% ±6%      3.6%
     rule (trend)      3.0%  150/189   79% ±6%      3.6%
     rule (trend)      5.0%  189/189  100% ±0%    100.0%
gradient boosting      0.5%   72/189   38% ±7%      0.5%
gradient boosting      1.0%   97/189   51% ±7%      1.0%
gradient boosting      2.0%  129/189   68% ±7%      2.0%
gradient boosting      3.0%  144/189   76% ±6%      3.0%
gradient boosting      5.0%  150/189   79% ±6%      5.0%
              GRU      0.5%   66/189   35% ±7%      0.5%
              GRU      1.0%  10

In [ ]:
class LSTMAE(nn.Module):
    def __init__(self, n_feat, mean, std, hidden=32, latent=8):
        super().__init__()
        self.register_buffer("mu", torch.tensor(mean, dtype=torch.float32))
        self.register_buffer("sd", torch.tensor(std, dtype=torch.float32))
        self.enc = nn.LSTM(n_feat, hidden, batch_first=True)
        self.to_latent = nn.Linear(hidden, latent)
        self.from_latent = nn.Linear(latent, hidden)
        self.dec = nn.LSTM(hidden, hidden, batch_first=True)
        self.out = nn.Linear(hidden, n_feat)

    def score(self, x):
        z = torch.clamp((x - self.mu) / self.sd, -10.0, 10.0)
        _, (h, _) = self.enc(z)
        lat = self.to_latent(h[-1])
        seq = self.from_latent(lat).unsqueeze(1).repeat(1, z.shape[1], 1)
        d, _ = self.dec(seq)
        rec = self.out(d)
        return ((rec - z) ** 2).mean(dim=(1, 2))      # reconstruction error per window

def ae_scores(model, idx, bs=8192):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(idx), bs):
            out.append(model.score(torch.from_numpy(X[idx[i:i + bs]])).numpy())
    return np.concatenate(out)

In [ ]:
AE_EPOCHS, AE_SAMPLE, AE_BATCH = 8, 150_000, 512
healthy_mask = ~np.isin(serial, list(failed_set))

oof_ae = np.zeros(len(y), dtype=np.float32)
for k in range(N_FOLDS):
    t0 = time.time()
    torch.manual_seed(SEED + k)
    rk = np.random.default_rng(SEED + k)
    tr_h = np.where((fold != k) & healthy_mask)[0]       # healthy drives only, other folds
    te = np.where(fold == k)[0]

    sample = X[rk.choice(tr_h, size=100_000, replace=False)]
    mean = sample.reshape(-1, F).mean(axis=0)
    std = sample.reshape(-1, F).std(axis=0)
    std[std == 0] = 1.0
    del sample

    model = LSTMAE(F, mean, std)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for ep in range(AE_EPOCHS):
        model.train()
        order = rk.choice(tr_h, size=min(AE_SAMPLE, len(tr_h)), replace=False)
        total = 0.0
        for i in range(0, len(order), AE_BATCH):
            b = order[i:i + AE_BATCH]
            opt.zero_grad()
            loss = model.score(torch.from_numpy(X[b])).mean()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item() * len(b)
        print(f"fold {k} epoch {ep + 1} | loss {total / len(order):.4f}")

    oof_ae[te] = ae_scores(model, te)
    print(f"fold {k} done in {time.time() - t0:.0f}s")

In [ ]:
t, ap = oof_report("LSTM autoencoder", oof_ae)
print(t.to_string(index=False))
print("PR-AUC:", round(float(ap), 4))
print("median error | healthy windows:", float(np.median(oof_ae[healthy_mask])),
      "| last-7-days windows of failed drives:", float(np.median(oof_ae[y == 1])))